## Bengali News Data Cleaning & Transformation
This notebook executes a complete data preparation pipeline for **Bengali**, transforming raw news articles into a structured, sentence-level dataset ready for zero-shot translation inference. This mirrors the preprocessing steps used for Odia and German, enabling a direct performance comparison between a high-resource Indian language (Bengali) and a low-resource one (Odia).

### Key Achievements:
* **Data Ingestion:**
  * Loaded the **Bengali Newspaper Dataset** from [Kaggle](https://www.kaggle.com/datasets/furcifer/bangla-newspaper-dataset), consisting of over **400k+ records**.
  * Successfully handled the JSON format, extracting relevant text fields while filtering out missing or malformed entries.
* **Linguistic Segmentation (IndicNLP):**
  * Utilized the `indic-nlp-library` (specifically for Bengali) to accurately split complex paragraphs into individual sentences.
  * This segmentation handles Bengali-specific punctuation (like the danda `|`) significantly better than standard regex splitting.
* **Zero-Shot Inference Readiness:**
  * **Prefix Injection:** Automatically injected the prompt `"translate Bengali to German: "` into every sentence.
  * **Filtering:** Applied rigorous length constraints (min 5 words, max 100 words) to remove noise, resulting in a high-quality inference file (`bengali_news_sentences_inference.jsonl`).
  * **Sampling:** Created a random subset of **50 sentences** specifically for model benchmarking.

### Workflow Context:
* **Source:** [data.json](https://www.kaggle.com/datasets/furcifer/bangla-newspaper-dataset/data) (Raw Bengali News)
* **Process:** Paragraph Splitting $\rightarrow$ Filtering $\rightarrow$ Prefixing
* **Output:** `bengali_news_sentences_inference.jsonl` (Ready for Zero-Shot Testing)

### Citation
```bibtex
@misc{zabir_al_nazi_2020,
	title={Bangla Newspaper Dataset},
	url={https://www.kaggle.com/dsv/1576225},
	DOI={10.34740/KAGGLE/DSV/1576225},
	publisher={Kaggle},
	author={Zabir Al Nazi},
	year={2020}
}
```

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# Install the necessary libraries for streaming JSON and Indic NLP
!pip install -q ijson indic-nlp-library

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.0/149.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 8.3 MB/s eta 0:00:00


In [ ]:
import ijson
import json
import os
import re
from tqdm import tqdm
from indicnlp.tokenize import sentence_tokenize

In [ ]:
# --- CONFIGURATION ---
INPUT_FILE = "/content/drive/MyDrive/Research_Paper_Publication/test/data/new_data/bangla_news_data.json"
OUTPUT_FILE = "/content/drive/MyDrive/Research_Paper_Publication/test/data/new_data/cleaned/bengali_sentences_inference_50.jsonl"
LANGUAGE_CODE = 'bn'  # Bengali language code for IndicNLP
PREFIX_BEN_TO_DEU = "translate Bengali to German: "
MAX_SENTENCES_TO_EXTRACT = 50
MIN_WORDS = 5
MAX_WORDS = 100

In [ ]:
# ==============================
# HELPER FUNCTION: TEXT CLEANING
# ==============================
def clean_text_fast(text):
    """
    Normalizes input text by sanitizing whitespace and control characters.

    This utility performs three specific cleaning operations:
    1. Replaces newlines (`\\n`), tabs (`\\t`), and carriage returns (`\\r`) with single spaces.
    2. Collapses multiple consecutive spaces into a single space using regex.
    3. Strips leading and trailing whitespace.

    Args:
        text (str or None): The raw input string to be processed.

    Returns:
        str: The normalized string. Returns an empty string if the input
             is None or empty.
    """
    if not text:
        return ""
    # 1. Replace explicit newlines/tabs/returns with a single space
    text = text.replace('\n', ' ').replace('\t', ' ').replace('\r', ' ')
    # 2. Collapse multiple spaces (2 or more) into a single space
    text = re.sub(r' {2,}', ' ', text)
    # 3. Strip leading and trailing whitespace
    return text.strip()

In [ ]:
# ==========================================
# 1. STREAMING JSON READER & FIELD DISCOVERY
# ==========================================

def discover_fields_and_process(input_file, output_file):
    """
    Stream-processes a large JSON dataset to discover structure and extract sentence-level data.

    This function uses `ijson` for memory-efficient parsing of large files. It operates
    in two distinct phases:
    1. **Schema Discovery**: Scans the first 100 records to identify available JSON keys
       and interactively prompts the user to select the key containing the text content.
    2. **Extraction & Splitting**: Re-reads the file, cleans paragraphs, splits them into
       sentences using IndicNLP, applies length filters, and formats them for training.

    Args:
        input_file (str): Path to the source JSON file.
        output_file (str): Path where the processed line-delimited JSON will be saved.

    Global Dependencies:
        - `MAX_SENTENCES_TO_EXTRACT` (int): Limit for total sentences.
        - `PREFIX_BEN_TO_DEU` (str): Task prefix for the model input.
        - `LANGUAGE_CODE` (str): ISO code for IndicNLP splitting (e.g., 'bn').
        - `MIN_WORDS` / `MAX_WORDS` (int): Filter thresholds for sentence length.
        - `ijson`, `tqdm`, `sentence_tokenize`: Required libraries.

    Returns:
        List[dict]: A list of processed sentence records. Returns an empty list
        or stops early if file errors occur or the text key is missing.
    """
    print(f"Reading from: {input_file}")

    # Use a dictionary to store all encountered field names
    field_counts = {}

    # We will assume the data structure is an array of objects: [{}, {}, ...]
    # If the JSON file is one large object {"data": [{}, {}]}, we'd change "item" to "data.item"
    JSON_PREFIX = "item"

    # --- STEP 1: Discover Field Names and Data Structure ---
    print("\n--- STEP 1: Discovering JSON Structure and Field Names (Checking first 100 records) ---")

    try:
        with open(input_file, "rb") as infile:
            # Check a small number of records to determine the key containing the Bengali text
            records_to_check = 100
            checked_count = 0

            for record in ijson.items(infile, JSON_PREFIX):
                if checked_count >= records_to_check:
                    break

                # Handle potential list wrapper (like in your 01_ notebook)
                obj = record[0] if isinstance(record, list) and len(record) > 0 else record

                if isinstance(obj, dict):
                    for key in obj.keys():
                        field_counts[key] = field_counts.get(key, 0) + 1
                    checked_count += 1

        print(f"Found {len(field_counts)} unique field names in the first {checked_count} records:")
        for field, count in field_counts.items():
            print(f"- {field}: Found {count} times")

    except Exception as e:
        print(f"Error during field discovery: {e}")
        return

    # --- STEP 2: Process and Split into Target Sentences ---
    # Let's assume the text field is named 'text' or similar.
    # **MANUAL CHECK REQUIRED HERE:** We need to confirm the key based on the output of Step 1.
    TEXT_KEY = input("Based on the fields above, please type the key name that contains the Bengali paragraph text (e.g., 'text', 'content'): ")
    if not TEXT_KEY:
        print("Text key not specified. Aborting processing.")
        return

    print(f"\n--- STEP 2: Extracting and Splitting Sentences using key: '{TEXT_KEY}' ---")
    processed_sentences = []
    sentence_id = 1
    total_paragraphs_read = 0

    try:
        # Re-open the file to start reading from the beginning
        with open(input_file, "rb") as infile, open(output_file, "w", encoding="utf-8") as outfile:

            # Use tqdm to monitor the processing of paragraphs
            for record in tqdm(ijson.items(infile, JSON_PREFIX), desc="Paragraphs Processed"):

                # Stop if we have enough sentences
                if sentence_id > MAX_SENTENCES_TO_EXTRACT:
                    break

                # Handle potential list wrapper
                obj = record[0] if isinstance(record, list) and len(record) > 0 else record

                if isinstance(obj, dict):
                    paragraph = obj.get(TEXT_KEY, "")
                    source = obj.get("source", "unknown") # Use 'source' if available

                    if not isinstance(paragraph, str) or not paragraph.strip():
                        continue

                    # 1. Clean the paragraph text
                    cleaned_paragraph = clean_text_fast(paragraph)

                    # 2. Split Paragraph into Sentences using IndicNLP
                    sentences = sentence_tokenize.sentence_split(cleaned_paragraph, lang=LANGUAGE_CODE)

                    for sent in sentences:
                        sent = sent.strip()

                        # 3. Filter and Prepare the Final Record
                        word_count = len(sent.split())
                        if MIN_WORDS <= word_count <= MAX_WORDS:
                            if sentence_id <= MAX_SENTENCES_TO_EXTRACT:

                                new_record = {
                                    "id": sentence_id,
                                    "input_text": PREFIX_BEN_TO_DEU + sent, # Add the prefix
                                    "raw_bengali": sent,
                                    "source": source,
                                    "original_paragraph_snippet": cleaned_paragraph[:50] + "..."
                                }

                                # 4. Write to output file
                                outfile.write(json.dumps(new_record, ensure_ascii=False) + '\n')
                                processed_sentences.append(new_record)
                                sentence_id += 1

                    total_paragraphs_read += 1

    except Exception as e:
        print(f"\nCRITICAL ERROR during sentence processing: {e}")

    print(f"\n✅ Finished processing.")
    print(f"Total paragraphs read: {total_paragraphs_read}")
    print(f"Created {len(processed_sentences)} individual sentence records (Target: {MAX_SENTENCES_TO_EXTRACT}).")
    print(f"Saved to: {output_file}")

    return processed_sentences

In [ ]:
# --- EXECUTE THE FUNCTION ---
final_data = discover_fields_and_process(INPUT_FILE, OUTPUT_FILE)

Reading from: /content/drive/MyDrive/Research_Paper_Publication/test/data/new_data/bangla_news_data.json

--- STEP 1: Discovering JSON Structure and Field Names (Checking first 100 records) ---
Found 10 unique field names in the first 100 records:
- author: Found 100 times
- category: Found 100 times
- category_bn: Found 100 times
- published_date: Found 100 times
- modification_date: Found 100 times
- tag: Found 100 times
- comment_count: Found 100 times
- title: Found 100 times
- url: Found 100 times
- content: Found 100 times
Based on the fields above, please type the key name that contains the Bengali paragraph text (e.g., 'text', 'content'): content

--- STEP 2: Extracting and Splitting Sentences using key: 'content' ---


Paragraphs Processed: 3it [00:00, 1004.30it/s]


✅ Finished processing.
Total paragraphs read: 3
Created 50 individual sentence records (Target: 50).
Saved to: /content/drive/MyDrive/Research_Paper_Publication/test/data/new_data/cleaned/bengali_sentences_inference_50.jsonl


In [ ]:
final_data

[{'id': 1,
  'input_text': 'translate Bengali to German: গাজীপুরের কালিয়াকৈর উপজেলার তেলিরচালা এলাকায় আজ বৃহস্পতিবার রাতের টিফিন খেয়ে একটি পোশাক কারখানার ৫০০ শ্রমিক অসুস্থ হয়ে পড়েছেন।',
  'raw_bengali': 'গাজীপুরের কালিয়াকৈর উপজেলার তেলিরচালা এলাকায় আজ বৃহস্পতিবার রাতের টিফিন খেয়ে একটি পোশাক কারখানার ৫০০ শ্রমিক অসুস্থ হয়ে পড়েছেন।',
  'source': 'unknown',
  'original_paragraph_snippet': 'গাজীপুরের কালিয়াকৈর উপজেলার তেলিরচালা এলাকায় আজ ...'},
 {'id': 2,
  'input_text': 'translate Bengali to German: এ ঘটনায় বিক্ষোভ করেছেন ওই কারখানার শ্রমিকেরা।',
  'raw_bengali': 'এ ঘটনায় বিক্ষোভ করেছেন ওই কারখানার শ্রমিকেরা।',
  'source': 'unknown',
  'original_paragraph_snippet': 'গাজীপুরের কালিয়াকৈর উপজেলার তেলিরচালা এলাকায় আজ ...'},
 {'id': 3,
  'input_text': 'translate Bengali to German: সফিপুর মডার্ন হাসপাতালের জরুরি বিভাগের চিকিত্সক আল আমিন প্রথম আলো ডটকমকে বলেন, খাদ্যে বিষক্রিয়ায় তাঁরা (শ্রমিকেরা) অসুস্থ হয়ে পড়েছেন।',
  'raw_bengali': 'সফিপুর মডার্ন হাসপাতালের জরুরি বিভাগের চিকি